# Preprocessing & Pelabelan Awal — IndoRoBERTa

**Notebook:** `02_preprocessing_and_labeling.ipynb`  
**Peneliti:** [Nama Peneliti]  
**Tanggal:** [Tanggal Pengerjaan]  

---

## Deskripsi

Notebook ini menjalankan dua proses utama:

1. **Preprocessing teks** menggunakan pipeline yang telah dirancang khusus untuk penelitian analisis sentimen Program Makan Bergizi Gratis (MBG). Pipeline menghasilkan dua varian teks:
   - `text_bert`: Teks yang dioptimalkan untuk model berbasis Transformer (IndoRoBERTa, IndoBERT+CNN) — berhenti pada tahap normalisasi tanpa stemming/stopword removal.
   - `text_lda`: Teks yang dioptimalkan untuk pemodelan topik LDA — dilanjutkan dengan stopword removal, stemming (PySastrawi), dan filtering.

2. **Pelabelan awal otomatis** menggunakan model **`w11wo/indonesian-roberta-base-sentiment-classifier`** (IndoRoBERTa) sebagai *anotator otomatis* untuk menghasilkan label pseudo-ground-truth. Label ini kemudian divalidasi secara manual oleh peneliti dan digunakan untuk menghitung **Cohen's Kappa** sebagai metrik Inter-Annotator Agreement (IAA).

---

## Justifikasi Pemilihan Model IndoRoBERTa

Model `w11wo/indonesian-roberta-base-sentiment-classifier` dipilih berdasarkan beberapa pertimbangan:

| Kriteria | Keterangan |
|---|---|
| **Arsitektur** | RoBERTa (Robustly Optimized BERT) — varian BERT dengan training lebih lama, batch lebih besar, dan tanpa NSP, terbukti superior untuk klasifikasi teks |
| **Bahasa** | Dilatih pada korpus bahasa Indonesia |
| **Task** | Fine-tuned langsung untuk sentiment classification (Positif/Negatif/Netral) |
| **Relevansi** | Label output konsisten dengan 3 kelas yang digunakan dalam penelitian ini |
| **Referensi** | Tersedia di Hugging Face Model Hub dan telah digunakan dalam beberapa penelitian NLP bahasa Indonesia |

**Peran dalam penelitian**: IndoRoBERTa berfungsi sebagai *anotator ke-2* (pseudo-annotator) untuk keperluan perhitungan Inter-Annotator Agreement (IAA) menggunakan Cohen's Kappa (κ). Nilai κ ≥ 0.61 (substantial agreement) menjadi threshold validasi kualitas pelabelan manual.

**Referensi**:  
- Liu, Y., et al. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach. *arXiv:1907.11692*.  
- Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37–46.

## 0. Setup: Mount Drive & Instalasi Library

Mount Google Drive untuk akses data dan instalasi library yang diperlukan. `PySastrawi` tidak tersedia secara default di Colab sehingga perlu diinstal secara eksplisit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install library tambahan
!pip install PySastrawi transformers torch -q

import os, re, requests, warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
warnings.filterwarnings('ignore')
tqdm.pandas()

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'GPU tersedia : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device   : {torch.cuda.get_device_name(0)}')
print(f'PyTorch      : {torch.__version__}')

## 1. Konfigurasi Path & Parameter

Seluruh path dan parameter penelitian didefinisikan terpusat di sel ini untuk memudahkan modifikasi.

In [ ]:
# ── PATH ─────────────────────────────────────────────────────────────────────
BASE_DIR     = '/content/drive/My Drive/skripsi/dataset/mbg'
PATH_KAMUS   = f'{BASE_DIR}/preprocessing/kamus'
INPUT_PATH   = f'{BASE_DIR}/processed/mbg_sampled.csv'
OUTPUT_PATH  = f'{BASE_DIR}/processed/mbg_labeled.csv'
OUTPUT_DIR   = f'{BASE_DIR}/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── PARAMETER ────────────────────────────────────────────────────────────────
RANDOM_SEED       = 42
BATCH_SIZE        = 32       # Sesuaikan dengan VRAM GPU Colab (T4: 32, P100: 64)
MAX_LENGTH        = 128      # Konsisten dengan training IndoBERT+CNN
ROBERTA_MODEL     = 'w11wo/indonesian-roberta-base-sentiment-classifier'

# Mapping label model → label penelitian (3 kelas)
LABEL_MAP = {
    'positive' : 'positive',
    'neutral'  : 'neutral',
    'negative' : 'negative',
    'POSITIVE' : 'positive',
    'NEUTRAL'  : 'neutral',
    'NEGATIVE' : 'negative',
    'LABEL_0'  : 'negative',   # fallback jika model pakai numeric label
    'LABEL_1'  : 'neutral',
    'LABEL_2'  : 'positive',
}

print('Konfigurasi berhasil dimuat.')
print(f'Model IndoRoBERTa : {ROBERTA_MODEL}')
print(f'Input             : {INPUT_PATH}')
print(f'Output            : {OUTPUT_PATH}')

## 2. Load Kamus & Resource Bahasa

Memuat seluruh kamus custom yang digunakan dalam pipeline preprocessing:

| File | Deskripsi |
|---|---|
| `demoji_code_mbg.csv` | Pemetaan emoji → frasa bahasa Indonesia |
| `akun_x_mbg.csv` | Pemetaan mention akun penting → nama asli |
| `kamus_alay_mbg.csv` | Kamus slang/alay custom domain MBG |
| `whitelist_hashtag_mbg.csv` | Hashtag relevan MBG yang dipertahankan dan dikonversi |
| `additional_stopwords_mbg.csv` | Stopword tambahan domain MBG untuk pipeline LDA |

Kamus slang digabungkan dengan kamus Nasal (colloquial Indonesian lexicon) dimana kamus custom **override** kamus Nasal untuk kata-kata yang spesifik domain MBG.

In [ ]:
def load_dict(filename):
    path = os.path.join(PATH_KAMUS, filename)
    if os.path.exists(path):
        return pd.read_csv(path, on_bad_lines='skip')
    print(f'  ⚠️  File tidak ditemukan: {filename}')
    return pd.DataFrame(columns=['original_term', 'replacement'])

print('Memuat kamus custom...')
df_demoji = load_dict('demoji_code_mbg.csv')
df_akun   = load_dict('akun_x_mbg.csv')
df_alay   = load_dict('kamus_alay_mbg.csv')
df_wl     = load_dict('whitelist_hashtag_mbg.csv')

# Load kamus Nasal (colloquial Indonesian lexicon)
print('Memuat kamus Nasal...')
!wget -q -N https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv
df_nasal = pd.read_csv('colloquial-indonesian-lexicon.csv')

# Konversi ke dict
whitelist_hashtags = set(
    df_wl['hashtag'].astype(str).str.lower().str.strip()
) if not df_wl.empty else set()

dict_demoji = {
    str(k).strip(): str(v).strip()
    for k, v in zip(df_demoji.get('original_term', []), df_demoji.get('replacement', []))
    if pd.notna(k) and pd.notna(v) and str(k).strip() != ''
}

dict_akun = {
    str(k).strip(): str(v).strip()
    for k, v in zip(df_akun.get('original_term', []), df_akun.get('replacement', []))
    if pd.notna(k) and pd.notna(v)
}

# Merge kamus alay: Nasal sebagai base, custom sebagai override
dict_nasal_raw = dict(zip(df_nasal['slang'], df_nasal['formal']))
dict_custom    = {
    str(k).lower(): str(v).lower()
    for k, v in zip(df_alay.get('original_term', []), df_alay.get('replacement', []))
    if pd.notna(k) and pd.notna(v)
}
full_dict_alay = {str(k).lower(): str(v).lower() for k, v in dict_nasal_raw.items()}
full_dict_alay.update(dict_custom)  # custom override nasal

# Pisahkan frasa (mengandung spasi) dan kata tunggal
dict_phrases   = {k: v for k, v in full_dict_alay.items() if ' ' in k}
dict_words     = {k: v for k, v in full_dict_alay.items() if ' ' not in k}
sorted_phrases = sorted(dict_phrases.keys(), key=len, reverse=True)

# Setup stopwords untuk LDA
print('Memuat stopwords...')
try:
    r = requests.get(
        'https://raw.githubusercontent.com/masdevid/ID-Stopwords/master/id.stopwords.02.01.2016.txt',
        timeout=10
    )
    list_stopwords = set(r.text.splitlines())
except Exception:
    from nltk.corpus import stopwords
    list_stopwords = set(stopwords.words('indonesian'))

path_add_stop = os.path.join(PATH_KAMUS, 'additional_stopwords_mbg.csv')
if os.path.exists(path_add_stop):
    df_stop   = pd.read_csv(path_add_stop, header=None)
    add_stops = set(df_stop[0].astype(str).str.lower().str.strip().tolist())
    list_stopwords.update(add_stops)
    print(f'  ✔️  Stopwords custom: {len(add_stops)} kata')

# Setup Sastrawi stemmer untuk LDA
print('Memuat Sastrawi stemmer...')
factory = StemmerFactory()
stemmer = factory.create_stemmer()

print(f'\n✅ Resource bahasa berhasil dimuat.')
print(f'   Emoji mapping     : {len(dict_demoji)} entri')
print(f'   Akun mapping      : {len(dict_akun)} entri')
print(f'   Whitelist hashtag : {len(whitelist_hashtags)} tag')
print(f'   Kamus alay total  : {len(full_dict_alay)} entri ({len(dict_phrases)} frasa, {len(dict_words)} kata)')
print(f'   Stopwords total   : {len(list_stopwords)} kata')

## 3. Pipeline Preprocessing

Pipeline preprocessing dirancang dalam 7 tahap modular. Setiap tahap memiliki fungsi tersendiri dan dapat diuji secara independen.

**Divergensi pipeline** terjadi setelah Step 4 (Normalization):
- `text_bert` → **berhenti di Step 4** — teks natural untuk Transformer
- `text_lda`  → **dilanjutkan ke Step 5–7** — teks bersih untuk topic modeling

| Step | Nama | BERT | LDA |
|---|---|---|---|
| 1 | Case Folding | ✅ | ✅ |
| 2 | Cleaning | ✅ | ✅ |
| 3 | Tokenization (parsing helper) | ✅ | ✅ |
| 4 | Normalization | ✅ → **OUTPUT** | ✅ |
| 5 | Stopword Removal | ❌ | ✅ |
| 6 | Stemming (PySastrawi) | ❌ | ✅ |
| 7 | Filtering | ❌ | ✅ → **OUTPUT** |

> **Catatan**: Stemming dan Stopword Removal **tidak diterapkan** pada `text_bert` karena IndoBERT/IndoRoBERTa dilatih dengan teks bahasa Indonesia yang natural. Penerapan stemming akan merusak subword tokenization model dan berpotensi menghasilkan token OOV (Out-of-Vocabulary).

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  STEP 1 — CASE FOLDING & HTML UNESCAPE
# ═══════════════════════════════════════════════════════════════
def step1_casefolding(text):
    if pd.isna(text) or str(text).strip() == '': return ''
    temp = str(text)
    temp = temp.replace('&amp;', ' dan ')
    temp = temp.replace('&lt;',  ' kurang dari ')
    temp = temp.replace('&gt;',  ' lebih dari ')
    temp = temp.replace('&quot;', '"')
    temp = temp.replace('&#39;', "'")
    return temp.lower()


# ═══════════════════════════════════════════════════════════════
#  STEP 2 — CLEANING
# ═══════════════════════════════════════════════════════════════
def step2_cleaning(text):
    if not text: return ''
    temp = text

    # 2.1 Emotikon teks
    temp = re.sub(r'[:;=x][\-~]?[)d\]]+', ' senang ', temp)
    temp = re.sub(r'[:;=x][\-~]?[(\[]+',  ' sedih ', temp)

    # 2.2 Hapus URL
    temp = re.sub(r'http\S+|www\S+|https\S+', ' ', temp, flags=re.MULTILINE)

    # 2.3 Singkatan dalam kurung yang tidak relevan
    for pat in [r'\(\s*mbg\s*\)', r'\(\s*bgn\s*\)', r'\(\s*apbn\s*\)', r'\(\s*as\s*\)']:
        temp = re.sub(pat, ' ', temp)

    # 2.4 Normalisasi whitespace & newline
    temp = temp.replace('\n', ' . ')
    temp = re.sub(r'[\t\xa0]', ' ', temp)
    temp = re.sub(r'([?!,.])(?=[a-z#])', r'\1 ', temp)

    # 2.5 Normalisasi suffix slang (otak'nya -> otaknya)
    temp = re.sub(r"(?<=[a-z])['`]+[ya]\b", 'nya', temp)

    # 2.6 Penyelamat huruf tunggal (s-nya -> huruf s nya)
    temp = re.sub(r'\b([a-z])(?:-|\'|\s)*nya\b', r'huruf \1 nya', temp)

    # 2.7 Pemangkas repetisi huruf (enaaaaak -> enak)
    temp = re.sub(r'([a-z])\1{2,}', r'\1', temp)

    # 2.8 Hashtag: whitelist dipertahankan & dikonversi, sisanya hapus
    for tag in re.findall(r'#\w+', temp):
        if tag.lower().lstrip('#') in whitelist_hashtags or tag.lower() in whitelist_hashtags:
            temp = temp.replace(tag, ' ' + tag[1:] + ' ')
        else:
            temp = temp.replace(tag, ' ')

    # 2.9 Mention penting → nama asli; sisanya hapus
    for k, v in dict_akun.items():
        temp = re.sub(re.escape(k), ' ' + v + ' ', temp, flags=re.IGNORECASE)
    temp = re.sub(r'@\w+', ' ', temp)

    # 2.10 Emoji → frasa (kamus custom)
    for k, v in dict_demoji.items():
        temp = temp.replace(k, ' ' + v + ' ')

    # 2.11 Normalisasi kata ulang angka (anak2 -> anak anak)
    temp = re.sub(r'\b([a-z]{2,})2(nya|ny|ku|mu|an|in|kan|san|kah|lah|pun|)\b',
                  r'\1 \1\2', temp)

    # 2.12 Currency shield
    temp = re.sub(r'\b(?:rp|rupiah)\s*[.,]?\s*(\d)', r'rp \1', temp)
    def ubah_mata_uang(m):
        angka  = m.group(1).replace(' ', ',')
        suffix = m.group(2)
        peta   = {'k':'ribu','rb':'ribu','ribu':'ribu','jt':'juta','juta':'juta',
                  'm':'miliar','miliar':'miliar','milyar':'miliar',
                  't':'triliun','triliun':'triliun','triliyun':'triliun','perak':'rupiah'}
        return f"{angka} {peta[suffix]}"
    temp = re.sub(
        r'\b(\d+(?:[.,\s]\d+)?)\s*(k|rb|ribu|jt|juta|m|miliar|milyar|t|triliun|triliyun|perak)\b',
        ubah_mata_uang, temp
    )
    temp = re.sub(r'\b(\d{1,3}(?:\.\d{3})+)\b',
                  lambda m: m.group(1).replace('.', ''), temp)

    # 2.13 Hapus format tanggal numerik (DD/MM/YYYY) — tidak dikonversi ke kata
    temp = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', ' ', temp)
    bulan_re = r'(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)'
    temp = re.sub(rf'\b\d{{1,2}}\s+{bulan_re}\s+\d{{2,4}}\b', ' tanggal ', temp)
    temp = re.sub(r'\b\d{1,2}[:.\-]\d{2}(?:[:.\-]\d{2})?\b', ' ', temp)

    # 2.14 Pecahan & persen
    temp = re.sub(r'(?<!\w)1/2(?!\w)', ' setengah ', temp)
    temp = re.sub(r'(?<!\w)1/3(?!\w)', ' sepertiga ', temp)
    temp = re.sub(r'(?<!\w)1/4(?!\w)', ' seperempat ', temp)
    temp = re.sub(r'(?<!\w)3/4(?!\w)', ' tiga perempat ', temp)
    temp = re.sub(r'(?<!\w)(\d+)/(\d+)(?!\w)', r'\1 per \2', temp)
    temp = temp.replace('%', ' persen ')

    # 2.15 Normalisasi tanda baca berlebih
    temp = re.sub(r'\.{3,}', ' ... ', temp)
    temp = re.sub(r'(?<!\.)\.\.(?!\.)', ' . ', temp)
    temp = re.sub(r'[^a-z0-9\s\?\!\,\.]', ' ', temp)

    return re.sub(r'\s+', ' ', temp).strip()


# ═══════════════════════════════════════════════════════════════
#  STEP 3 — TOKENIZATION (parsing helper, bukan output)
# ═══════════════════════════════════════════════════════════════
def step3_tokenization(text):
    """Whitespace split sebagai parsing helper untuk normalisasi.
    Tidak digunakan sebagai output final — re-join setelah step 4."""
    if not text: return ''
    return ' '.join(text.split())


# ═══════════════════════════════════════════════════════════════
#  STEP 4 — NORMALIZATION
#  OUTPUT: text_bert (IndoRoBERTa, IndoBERT+CNN)
# ═══════════════════════════════════════════════════════════════
def step4_normalization(text):
    if not text: return ''
    temp = ' ' + text + ' '

    # 4.1 Normalisasi frasa (longest match first)
    for phrase in sorted_phrases:
        if phrase in temp:
            temp = re.sub(r'\b' + re.escape(phrase) + r'\b',
                          ' ' + dict_phrases[phrase] + ' ', temp)

    # 4.2 Normalisasi kata tunggal
    tokens           = temp.split()
    normalized       = [dict_words.get(w, w) for w in tokens]
    temp             = ' '.join(normalized)

    # 4.3 Normalisasi negasi (word boundary ketat)
    temp = re.sub(r'\bnggak\b', 'tidak', temp)
    temp = re.sub(r'\bngga\b',  'tidak', temp)
    temp = re.sub(r'\bgak\b',   'tidak', temp)
    temp = re.sub(r'\bga\b',    'tidak', temp)  # word boundary, aman dari 'gagal' dll

    # 4.4 Dedup frasa hasil ekspansi
    for p in ['makan bergizi gratis', 'badan gizi nasional',
              'anggaran pendapatan dan belanja negara']:
        temp = re.sub(rf'\b({re.escape(p)})(\s+\1)+\b', r'\1', temp)

    return re.sub(r'\s+', ' ', temp).strip()


# ═══════════════════════════════════════════════════════════════
#  STEP 5 — STOPWORD REMOVAL (LDA only)
# ═══════════════════════════════════════════════════════════════
def step5_stopword_removal(text):
    if not text: return ''
    temp   = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = temp.split()
    return ' '.join([w for w in tokens if w not in list_stopwords])


# ═══════════════════════════════════════════════════════════════
#  STEP 6 — STEMMING (LDA only, PySastrawi per-token)
# ═══════════════════════════════════════════════════════════════
def step6_stemming(text):
    """Stem per token — stemmer.stem() dirancang untuk satu kata."""
    if not text: return ''
    return ' '.join([stemmer.stem(w) for w in text.split()])


# ═══════════════════════════════════════════════════════════════
#  STEP 7 — FILTERING (LDA only)
#  OUTPUT: text_lda
# ═══════════════════════════════════════════════════════════════
def step7_filtering(text):
    """Hapus token < 3 karakter dan token berupa digit murni.
    Kata satuan angka (persen, ribu, juta, miliar) dipertahankan."""
    if not text: return ''
    tokens = text.split()
    return ' '.join([w for w in tokens if len(w) >= 3 and not w.isdigit()])


# ═══════════════════════════════════════════════════════════════
#  MASTER PIPELINE — return dict {text_bert, text_lda}
# ═══════════════════════════════════════════════════════════════
def preprocess(text):
    """Pipeline preprocessing lengkap.
    Returns dict dengan dua varian teks:
      - text_bert : untuk IndoRoBERTa & IndoBERT+CNN (step 1-4)
      - text_lda  : untuk LDA topic modeling (step 1-7)
    """
    s1 = step1_casefolding(text)
    s2 = step2_cleaning(s1)
    s3 = step3_tokenization(s2)
    s4 = step4_normalization(s3)    # ← text_bert berhenti di sini
    s5 = step5_stopword_removal(s4)
    s6 = step6_stemming(s5)
    s7 = step7_filtering(s6)        # ← text_lda
    return {'text_bert': s4, 'text_lda': s7}


print('✅ Pipeline preprocessing berhasil didefinisikan.')

## 4. Pengujian Pipeline Preprocessing

Menguji pipeline dengan 3 kasus representatif untuk memvalidasi setiap tahap sebelum diterapkan ke seluruh dataset. Pengujian ini penting untuk memastikan tidak ada bug dan hasil sesuai ekspektasi.

In [ ]:
test_cases = [
    "Anggaran MBG &amp; bansos cair barengan? &lt; 50% yg sampai desa. Keren bangeeeetttt =D",
    "program makan bergizi gratis (mbg) ini anggarannya rp.15.000 lho di tanggal 17/04/2025 \n keren banget...indonesia.#mbg",
    "hampir 50% dana dipotong? gila aja! 1/2 nya dikorupsi kali ya?!",
]

for i, tc in enumerate(test_cases, 1):
    result = preprocess(tc)
    print(f'🧪 KASUS {i}')
    print(f'  ORIGINAL  : {tc}')
    print(f'  TEXT_BERT : {result["text_bert"]}')
    print(f'  TEXT_LDA  : {result["text_lda"]}')
    print('─' * 70)

## 5. Terapkan Preprocessing ke Seluruh Dataset

Menerapkan pipeline preprocessing ke semua tweet dalam `mbg_sampled.csv`. Proses ini menghasilkan dua kolom baru: `text_bert` dan `text_lda`.

Setelah preprocessing, dilakukan **filtering akhir** untuk menghapus tweet yang menjadi terlalu pendek (≤ 3 kata) setelah proses normalisasi, serta tweet yang menghasilkan teks kosong.

In [ ]:
print('Memuat data sampled...')
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Data loaded: {len(df):,} baris')

# Terapkan pipeline preprocessing
print('\nMenerapkan preprocessing...')
results       = df['full_text'].progress_apply(preprocess)
df['text_bert'] = results.apply(lambda x: x['text_bert'])
df['text_lda']  = results.apply(lambda x: x['text_lda'])

n_before = len(df)

# Filter tweet yang text_bert terlalu pendek atau kosong
df['bert_word_count'] = df['text_bert'].str.split().str.len().fillna(0)
df = df[df['text_bert'].str.strip() != '']
df = df[df['bert_word_count'] > 3]

n_after = len(df)
df = df.reset_index(drop=True)

print(f'\nSebelum filter post-preprocessing : {n_before:,}')
print(f'Setelah filter (text_bert > 3 kata): {n_after:,} (-{n_before - n_after:,})')
print(f'\nContoh hasil preprocessing (3 baris):')
df[['full_text', 'text_bert', 'text_lda']].head(3)

## 6. Pelabelan Awal — IndoRoBERTa

Model `w11wo/indonesian-roberta-base-sentiment-classifier` digunakan untuk menghasilkan label sentimen otomatis beserta confidence score. Model ini berfungsi sebagai *anotator ke-2* dalam skema pseudo Inter-Annotator Agreement (IAA).

**Output per tweet:**
- `label_roberta`: Label prediksi (positive / negative / neutral)
- `confidence_roberta`: Confidence score (0.0–1.0) dari prediksi

Inference dilakukan dalam batch menggunakan GPU Colab untuk efisiensi waktu. Proses ini menggunakan `text_bert` (output step 4) sebagai input — konsisten dengan preprocessing yang akan digunakan pada model IndoBERT+CNN.

> **Catatan batasan model**: Confidence score yang tinggi tidak selalu menjamin label yang benar, terutama untuk tweet ambigu atau sarkastis. Oleh karena itu validasi manual oleh peneliti tetap menjadi referensi utama (ground truth).

In [ ]:
print(f'Memuat model: {ROBERTA_MODEL}')
print('Proses ini membutuhkan beberapa menit pada pertama kali...')

tokenizer_roberta = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model_roberta     = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)

sentiment_pipeline = pipeline(
    task            = 'sentiment-analysis',
    model           = model_roberta,
    tokenizer       = tokenizer_roberta,
    device          = DEVICE,
    max_length      = MAX_LENGTH,
    truncation      = True,
    padding         = True,
)

# Inspeksi label yang digunakan model
id2label = model_roberta.config.id2label
print(f'\n✅ Model berhasil dimuat.')
print(f'   Label model : {id2label}')
print(f'   Device      : {"GPU" if DEVICE == 0 else "CPU"}')

In [ ]:
def run_inference_batched(texts, batch_size=BATCH_SIZE):
    """Jalankan inference IndoRoBERTa dalam batch.
    Returns list of dict {'label': str, 'score': float}"""
    all_results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Inference IndoRoBERTa'):
        batch = texts[i:i + batch_size].tolist()
        try:
            preds = sentiment_pipeline(batch)
            all_results.extend(preds)
        except Exception as e:
            print(f'  ⚠️  Error pada batch {i//batch_size}: {e}')
            # Fallback: proses satu per satu jika batch gagal
            for text in batch:
                try:
                    all_results.append(sentiment_pipeline([text])[0])
                except Exception:
                    all_results.append({'label': 'neutral', 'score': 0.0})
    return all_results

print(f'Menjalankan inference pada {len(df):,} tweet (batch size: {BATCH_SIZE})...')
predictions = run_inference_batched(df['text_bert'])

# Ekstrak label dan confidence score
df['label_roberta_raw'] = [p['label'] for p in predictions]
df['confidence_roberta'] = [round(p['score'], 4) for p in predictions]

# Mapping ke label standar penelitian
df['label_roberta'] = df['label_roberta_raw'].map(LABEL_MAP).fillna('neutral')

print(f'\n✅ Inference selesai.')
print(f'\nDistribusi label IndoRoBERTa:')
print(df['label_roberta'].value_counts())
print(f'\nDistribusi confidence score:')
print(df['confidence_roberta'].describe().round(4))

## 7. Analisis Distribusi Confidence Score

Visualisasi distribusi confidence score penting untuk memahami tingkat kepastian model dalam melabeli data. Tweet dengan confidence rendah (< 0.60) umumnya merupakan kasus ambigu yang lebih membutuhkan perhatian khusus saat anotasi manual.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribusi confidence score keseluruhan
axes[0].hist(df['confidence_roberta'], bins=40, color='#534AB7',
             edgecolor='none', alpha=0.85)
axes[0].axvline(x=0.6, color='#E24B4A', linestyle='--', linewidth=1.2,
                label='Threshold 0.60')
axes[0].set_xlabel('Confidence score', fontsize=11)
axes[0].set_ylabel('Jumlah tweet', fontsize=11)
axes[0].set_title('Distribusi confidence score IndoRoBERTa', fontsize=12)
axes[0].legend(fontsize=10)
sns.despine(ax=axes[0])

# Plot 2: Distribusi label per confidence tier
df['conf_tier'] = pd.cut(
    df['confidence_roberta'],
    bins=[0, 0.6, 0.8, 1.0],
    labels=['Rendah (<0.60)', 'Sedang (0.60–0.80)', 'Tinggi (>0.80)']
)
tier_dist = df.groupby(['conf_tier', 'label_roberta']).size().unstack(fill_value=0)
tier_dist.plot(kind='bar', ax=axes[1], color=['#AFA9EC', '#E24B4A', '#5DCAA5'],
               edgecolor='none', width=0.65)
axes[1].set_xlabel('Confidence tier', fontsize=11)
axes[1].set_ylabel('Jumlah tweet', fontsize=11)
axes[1].set_title('Distribusi label per confidence tier', fontsize=12)
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Label', fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Ringkasan confidence tier
n_low  = (df['confidence_roberta'] < 0.6).sum()
n_high = (df['confidence_roberta'] >= 0.8).sum()
print(f'Tweet confidence rendah (<0.60) : {n_low:,} ({n_low/len(df)*100:.1f}%) — prioritas review manual')
print(f'Tweet confidence tinggi (≥0.80) : {n_high:,} ({n_high/len(df)*100:.1f}%)')

## 8. Cohen's Kappa — Inter-Annotator Agreement

Sel ini menghitung **Cohen's Kappa (κ)** sebagai metrik validasi kualitas pelabelan. Perhitungan dilakukan setelah peneliti melengkapi kolom `label` (anotasi manual) pada file output.

**Interpretasi nilai κ (Landis & Koch, 1977):**

| Nilai κ | Interpretasi |
|---|---|
| < 0.20 | Slight agreement |
| 0.21 – 0.40 | Fair agreement |
| 0.41 – 0.60 | Moderate agreement |
| **0.61 – 0.80** | **Substantial agreement** ← target minimum |
| 0.81 – 1.00 | Almost perfect agreement |

**Referensi**: Landis, J. R., & Koch, G. G. (1977). The measurement of observer agreement for categorical data. *Biometrics*, 33(1), 159–174.

In [ ]:
# ── Jalankan sel ini SETELAH kolom 'label' diisi secara manual ────────────────

def compute_iaa(df_labeled):
    """Hitung Cohen's Kappa antara label manual dan label IndoRoBERTa.
    Hanya tweet yang sudah dilabeli manual yang dihitung."""
    # Filter hanya baris yang sudah dilabeli manual
    df_valid = df_labeled[
        df_labeled['label'].isin(['positive', 'negative', 'neutral']) &
        df_labeled['label_roberta'].isin(['positive', 'negative', 'neutral'])
    ].copy()

    if len(df_valid) == 0:
        print('⚠️  Belum ada label manual. Isi kolom "label" terlebih dahulu.')
        return

    kappa = cohen_kappa_score(df_valid['label'], df_valid['label_roberta'])

    # Interpretasi
    if kappa < 0.20:   interp = 'Slight'
    elif kappa < 0.40: interp = 'Fair'
    elif kappa < 0.60: interp = 'Moderate'
    elif kappa < 0.80: interp = 'Substantial ✅'
    else:              interp = 'Almost Perfect ✅'

    print('=' * 50)
    print(' INTER-ANNOTATOR AGREEMENT (IAA)')
    print('=' * 50)
    print(f' Jumlah data terlabeli  : {len(df_valid):,}')
    print(f' Cohen\'s Kappa (κ)      : {kappa:.4f}')
    print(f' Interpretasi           : {interp}')
    print(f' Target minimum (κ≥0.61): {"TERPENUHI ✅" if kappa >= 0.61 else "BELUM TERPENUHI ⚠️"}')
    print('=' * 50)

    # Confusion matrix IAA
    labels_order = ['positive', 'negative', 'neutral']
    cm = confusion_matrix(df_valid['label'], df_valid['label_roberta'],
                          labels=labels_order)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_order, yticklabels=labels_order, ax=ax)
    ax.set_xlabel('IndoRoBERTa (anotator otomatis)', fontsize=11)
    ax.set_ylabel('Manual (anotator manusia)', fontsize=11)
    ax.set_title(f'Confusion matrix IAA (κ = {kappa:.4f})', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/iaa_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nClassification report (manual vs roberta):')
    print(classification_report(df_valid['label'], df_valid['label_roberta'],
                                target_names=labels_order))
    return kappa

# Panggil setelah anotasi manual selesai:
# kappa = compute_iaa(df)
print('✅ Fungsi compute_iaa() siap.')
print('   Jalankan: kappa = compute_iaa(df)  setelah kolom "label" diisi manual.')

## 9. Export Output

Menyimpan dataset hasil preprocessing dan pelabelan IndoRoBERTa ke file CSV. File ini siap digunakan untuk:
1. **Anotasi manual** — isi kolom `label` dengan Positif/Negatif/Netral
2. **Perhitungan Cohen's Kappa** — jalankan `compute_iaa(df)` setelah anotasi selesai
3. **Training IndoBERT+CNN** — gunakan kolom `text_bert` dan `label` (ground truth)
4. **LDA topic modeling** — gunakan kolom `text_lda` dan `label` (ground truth)

In [ ]:
# Susun kolom output
output_cols = [
    'id_str', 'created_at', 'period',
    'full_text',
    'text_bert',
    'text_lda',
    'label',              # diisi manual (ground truth)
    'label_roberta',      # prediksi IndoRoBERTa
    'confidence_roberta', # confidence score
    'annotator_notes',
    'word_count',
    'bert_word_count',
    'username',
    'source_keyword',
    'lang',
]
output_cols_exist = [c for c in output_cols if c in df.columns]
df_out = df[output_cols_exist].copy()

# Simpan
df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 55)
print(' PREPROCESSING & LABELING SUMMARY')
print('=' * 55)
print(f' Data input              : {INPUT_PATH.split("/")[-1]}')
print(f' Total data diproses     : {len(df_out):,} tweet')
print(f' Model IndoRoBERTa       : {ROBERTA_MODEL}')
print(f' Distribusi label roberta:')
for lbl, cnt in df_out['label_roberta'].value_counts().items():
    print(f'   {lbl:12s}: {cnt:,} ({cnt/len(df_out)*100:.1f}%)')
print(f' Confidence mean         : {df_out["confidence_roberta"].mean():.4f}')
print(f' Output disimpan ke      : {OUTPUT_PATH}')
print('=' * 55)
print('\nLangkah selanjutnya:')
print('  1. Buka mbg_labeled.csv — isi kolom "label" secara manual')
print('  2. Jalankan kembali sel 8: kappa = compute_iaa(df)')
print('  3. Pastikan κ ≥ 0.61 sebelum lanjut ke training')
print('  4. Notebook 03: Training IndoBERT+CNN')

---

## Catatan Metodologis untuk Laporan

### Preprocessing Pipeline

Pipeline preprocessing dirancang dalam dua jalur yang divergen pada Step 4:

- **Jalur BERT** (Step 1–4): Menghasilkan teks natural yang mempertahankan struktur morfologi, stopword, dan konteks kalimat. Hal ini penting karena IndoBERT dan IndoRoBERTa menggunakan subword tokenizer (WordPiece/BPE) yang dilatih pada teks natural — penerapan stemming akan menghasilkan token yang tidak dikenal (*out-of-vocabulary*) dan merusak representasi kontekstual model.

- **Jalur LDA** (Step 1–7): Menghasilkan kata-kata bersih yang tereduksi ke bentuk dasar. LDA sebagai model berbasis frekuensi ko-kemunculan kata membutuhkan teks yang homogen: stopword dihapus agar tidak mendominasi topik, dan stemming dilakukan agar variasi morfologi ("mendukung", "dukungan", "didukung") diperlakukan sebagai satu konsep yang sama.

### Pelabelan dengan IndoRoBERTa sebagai Pseudo-Annotator

Penggunaan IndoRoBERTa sebagai anotator otomatis mengikuti pendekatan *machine-assisted annotation* yang telah divalidasi dalam literatur NLP. Model ini tidak menggantikan anotasi manual, melainkan berfungsi sebagai referensi pembanding untuk mengukur konsistensi anotator manusia melalui Cohen's Kappa (κ).

Pendekatan ini dipilih karena keterbatasan sumber daya untuk melibatkan lebih dari satu anotator manusia dalam penelitian skripsi, sehingga IAA dihitung antara anotator manusia (peneliti) dan model IndoRoBERTa sebagai *proxy annotator ke-2*.

### Limitasi

IndoRoBERTa tidak dilatih secara spesifik pada domain MBG atau Twitter, sehingga akurasinya pada teks informal dan domain-specific mungkin tidak optimal. Oleh karena itu, label ground truth penelitian ini menggunakan **anotasi manual oleh peneliti** sebagai referensi utama, bukan label IndoRoBERTa.